# NYC Public School Performance with Data Integration & Visualization

In [12]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import os

# Use a renderer that works reliably in PyCharm and Jupyter
pio.renderers.default = "browser"

# Load the processed data and try multiple paths for portability
for candidate in ["../data/processed/merged_school_data.csv",
                  "data/processed/merged_school_data.csv",
                  os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "merged_school_data.csv")]:
    if os.path.exists(candidate):
        data_path = candidate
        break
else:
    raise FileNotFoundError("Cannot find merged_school_data.csv â€” run data_processing.py first")

merged_df = pd.read_csv(data_path)
print(f"Loaded {len(merged_df)} records with {len(merged_df.columns)} columns")
print(f"Boroughs: {sorted(merged_df['borough'].unique())}")
print(f"School years: {sorted(merged_df['year'].dropna().unique())}")
merged_df.head()

Loaded 421 records with 70 columns
Boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
School years: ['2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20']


,dbn,school_name,school_type,enrollment,survey_pp_ri,survey_pp_ct,survey_pp_se,survey_pp_es,survey_pp_sf,survey_pp_tr,...,white_1,missing_race_ethnicity_data,missing_race_ethnicity_data_1,students_with_disabilities,students_with_disabilities_1,english_language_learners,english_language_learners_1,poverty,poverty_1,economic_need_index
0,01M292,Orchard Collegiate Academy,High School,226,0.74,0.80,0.71,0.88,0.86,0.89,...,0.161,1328.0,0.007,36271.0,0.203,19358.0,0.109,114277.0,0.641,0.598
1,01M448,University Neighborhood High School,High School,492,0.85,0.91,0.87,0.92,0.92,0.93,...,0.169,1246.0,0.007,38063.0,0.214,18102.0,0.102,118397.0,0.667,0.665
2,01M450,East Side Community School,High School,364,NaN,NaN,NaN,1.00,0.97,1.00,...,0.170,2004.0,0.011,38878.0,0.215,17095.0,0.095,119314.0,0.661,0.679
3,01M509,Marta Valle High School,High School,97,0.80,0.93,0.81,0.89,0.98,0.99,...,0.161,1328.0,0.007,36271.0,0.203,19358.0,0.109,114277.0,0.641,0.598
4,01M539,"New Explorations into Science, Technology and ...",High School,654,0.80,0.79,0.74,0.82,0.92,0.87,...,0.161,1328.0,0.007,36271.0,0.203,19358.0,0.109,114277.0,0.641,0.598


## 1. ELA Performance vs. Borough Economic Need Index

In [13]:
fig = px.scatter(
    merged_df.dropna(subset=["economic_need_index", "rating_ela_grade_8_pct_rs"]),
    x="economic_need_index",
    y="rating_ela_grade_8_pct_rs",
    color="borough",
    hover_data=["school_name", "dbn", "year"],
    title="ELA Performance (Grade 8) vs. Borough Economic Need Index",
    labels={
        "economic_need_index": "Borough Economic Need Index",
        "rating_ela_grade_8_pct_rs": "ELA Performance (Grade 8 % Meeting Standards)",
    },
    opacity=0.7,
)
fig.update_layout(template="plotly_white")
fig.show()

## 2. Average ELA Performance by Borough

In [14]:
borough_avg = (
    merged_df.dropna(subset=["rating_ela_grade_8_pct_rs"])
    .groupby("borough")["rating_ela_grade_8_pct_rs"]
    .mean()
    .reset_index()
    .sort_values("rating_ela_grade_8_pct_rs", ascending=False)
)
fig = px.bar(
    borough_avg,
    x="borough",
    y="rating_ela_grade_8_pct_rs",
    color="borough",
    title="Average ELA Performance (Grade 8) by Borough",
    labels={
        "borough": "Borough",
        "rating_ela_grade_8_pct_rs": "Average ELA Performance (Grade 8)",
    },
    text_auto=".2f",
)
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

## 3. Distribution of Economic Need Index by Borough

In [15]:
fig = px.box(
    merged_df.dropna(subset=["economic_need_index"]),
    x="borough",
    y="economic_need_index",
    color="borough",
    title="Distribution of Borough Economic Need Index",
    labels={
        "borough": "Borough",
        "economic_need_index": "Economic Need Index",
    },
    points="all",
)
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

## 4. Math vs. ELA Performance by School Type

In [16]:
fig = px.scatter(
    merged_df.dropna(subset=["rating_ela_grade_8_pct_rs", "rating_mth_grade_8_pct_rs"]),
    x="rating_ela_grade_8_pct_rs",
    y="rating_mth_grade_8_pct_rs",
    color="school_type",
    hover_data=["school_name", "borough"],
    title="Math vs. ELA Performance (Grade 8)",
    labels={
        "rating_ela_grade_8_pct_rs": "ELA Performance (Grade 8)",
        "rating_mth_grade_8_pct_rs": "Math Performance (Grade 8)",
    },
    opacity=0.7,
)
fig.update_layout(template="plotly_white")
fig.show()

## 5. Average Survey Scores by Borough (Heatmap)

In [17]:
survey_cols = ["survey_pp_ri", "survey_pp_ct", "survey_pp_se", "survey_pp_es", "survey_pp_sf", "survey_pp_tr"]
survey_labels = {
    "survey_pp_ri": "Rigorous Instruction",
    "survey_pp_ct": "Collaborative Teachers",
    "survey_pp_se": "Supportive Environment",
    "survey_pp_es": "Effective School Leadership",
    "survey_pp_sf": "Strong Family-Community Ties",
    "survey_pp_tr": "Trust",
}
available_survey_cols = [c for c in survey_cols if c in merged_df.columns]
if available_survey_cols:
    survey_by_borough = merged_df.groupby("borough")[available_survey_cols].mean()
    survey_by_borough = survey_by_borough.rename(columns=survey_labels)
    fig = px.imshow(
        survey_by_borough,
        title="Average Survey Scores by Borough",
        labels=dict(x="Survey Dimension", y="Borough", color="Score"),
        color_continuous_scale="YlGnBu",
        aspect="auto",
    )
    fig.update_layout(template="plotly_white")
    fig.show()

## 6. Borough Poverty Rate vs. Economic Need Index

In [18]:
demo_cols = ["borough", "year", "poverty_1", "economic_need_index"]
if all(c in merged_df.columns for c in demo_cols):
    demo_summary = merged_df.drop_duplicates(subset=["borough", "year"])[demo_cols].dropna()
    fig = px.scatter(
        demo_summary,
        x="poverty_1",
        y="economic_need_index",
        color="borough",
        symbol="year",
        title="Borough Poverty Rate vs. Economic Need Index (by Year)",
        labels={
            "poverty_1": "Poverty Rate",
            "economic_need_index": "Economic Need Index",
        },
    )
    fig.update_layout(template="plotly_white")
    fig.show()